In [1087]:
# import pandas as pd
# import os

# folder_path = 'D:/Raylab/LiMeNEx/FetchData/Physiologicalsystem.csv'

# df = pd.read_csv(folder_path)

# df['Tissue'] = [var.lower().replace(" ","").strip() for var in df['Tissue']]

# df.to_csv(folder_path)

In [1210]:
import pandas as pd
tempDf = pd.read_csv("D:/Raylab/LiMeNEx/FetchData/Physiologicalsystem.csv")
tempDf.head()
ActualTissue = set(list(tempDf['Tissue']))

In [1211]:
len(ActualTissue)

49

Merge SPP CSV

In [1212]:
import os
import pandas as pd

# Define the path to the folder containing CSV files
folderName=  'steroidHormoneSynthesis'
folder_path = f'D:/Raylab/LiMeNEx/FetchData/data/DikshaFiles/{folderName}'

# Create an empty list to store individual dataframes
df_list = []

# Loop through all the files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        # Construct the full file path
        file_path = os.path.join(folder_path, filename)
        
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path)
        
        # Append the DataFrame to the list
        df_list.append(df)

# Concatenate all dataframes in the list
combined_df = pd.concat(df_list, ignore_index=True)

# Now `combined_df` contains all the data from the CSV files concatenated together
combined_df.head()


,Database,TF,TargetGene,Tissue,Experiment
0,SPP,FOXA1,AKR1C1,prostate,"FOXA1 ChIP-Seq, CON KD | FOXA1 ChIP-Seq, CREB1..."
1,SPP,YY1,AKR1C1,lung,YY1 IP - A549 cells
2,SPP,FOXA1,AKR1C1,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, FOXA1 ChIP-Seq -..."
3,SPP,DEX,AKR1C1,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, DEX | FOXA1 ChIP..."
4,SPP,NFYB,AKR1C1,leukocytes,NFYB IP - K562 cells


In [1213]:
list(combined_df.columns)

['Database', 'TF', 'TargetGene', 'Tissue', 'Experiment']

In [1214]:
combined_df = combined_df.drop([list(combined_df.columns)[0],'Database'], axis = 1)
combined_df['Chea'] = ""
combined_df['Signor'] = ""
combined_df['Trrust'] = ""

ActualGenes = set(list(combined_df['TargetGene'].unique()))

In [1215]:
'CYP19A1' in ActualGenes

True

In [1216]:
combined_df.head()

,TF,TargetGene,Tissue,Experiment,Chea,Signor,Trrust
0,FOXA1,AKR1C1,prostate,"FOXA1 ChIP-Seq, CON KD | FOXA1 ChIP-Seq, CREB1...",,,
1,YY1,AKR1C1,lung,YY1 IP - A549 cells,,,
2,FOXA1,AKR1C1,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, FOXA1 ChIP-Seq -...",,,
3,DEX,AKR1C1,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, DEX | FOXA1 ChIP...",,,
4,NFYB,AKR1C1,leukocytes,NFYB IP - K562 cells,,,


In [1217]:
l = list(combined_df['Tissue'].unique())
" ".join(l)

'prostate lung mammarygland leukocytes stomach adiposetissue stemcells liver uterus skin vasculature pancreas hspcs embryonictissue colon cns skeletalmuscle megakaryocytelineage esophagus kidney heart miscellaneous bone sensory placenta'

Signor

In [1219]:
Signor_path = f'D:/Raylab/LiMeNEx/FetchData/data/DikshaFiles/{folderName}/Signor/{folderName} - Signor.csv'
SigDf = pd.read_csv(Signor_path)
SigDf.head()

,TargetGene,TF,Library/PMID,validated genes,Tissue,Paper Links
0,CYP11A1,GATA6,15284005,Yes,Ovary,https://pubmed.ncbi.nlm.nih.gov/15284005/
1,CYP11A1,FOXL2,15284005,Yes,Ovary,https://pubmed.ncbi.nlm.nih.gov/15284005/
2,CYP17A1,TEFB,33176151,Yes,Liver,https://pubmed.ncbi.nlm.nih.gov/33176151/
3,CYP17A1,GATA6,15284005,Yes,Liver,http://www.ncbi.nlm.nih.gov/pubmed/15284005
4,HSD3B2,JUN,19022561,Yes,Uterus,https://pubmed.ncbi.nlm.nih.gov/19022561/


In [1220]:
SigDf.columns

Index(['TargetGene', 'TF', 'Library/PMID', 'validated genes', 'Tissue',
       'Paper Links'],
      dtype='object')

In [1097]:
# SigDf['validated genes'] = [var.strip().lower() for var in list(SigDf['validated genes'])]
# SigDf = SigDf[(SigDf['validated genes'] == 'yes')]

In [1221]:
for i in range(0,len(SigDf)):
    tissueList = set([var.lower().replace(" ", "").strip() for var in SigDf.iloc[i]['Tissue'].split(',')])
    

    redundantTissue = set(tissueList) - ActualTissue
    if len(redundantTissue) != 0:
        print("found redundant Tissue :", redundantTissue)
        break

    # TF = SigDf.iloc[i]['TF']
    TF = set([var.replace(" ", "").strip() for var in SigDf.iloc[i]['TF'].split(',')])
    TargetGene = SigDf.iloc[i]['TargetGene']
    PMID = str(SigDf.iloc[i]['Library/PMID'])
    
    if TargetGene not in ActualGenes:
        print("Redundant Gene :",TargetGene)
        continue

    for tis in tissueList:
        for tf in TF:

            mask = (combined_df['TF'] == tf) & (combined_df['TargetGene'] == TargetGene) & (combined_df['Tissue'] == tis)
            temp = combined_df[mask]

            if(len(temp) != 0):
                if(len(temp) > 1):
                    print('Here')
                index = temp.index[0]
                # combined_df.at[index,'Signor'] += f",{str(PMID)}"
                if pd.isna(combined_df.at[index, 'Signor']) or combined_df.at[index, 'Signor'] == '':
                    combined_df.at[index, 'Signor'] = str(PMID)  # Assign PMID directly
                else:
                    combined_df.at[index, 'Signor'] += f";{str(PMID)}"  # Append with a comma

            else:
                newrow = {"TF":tf, 'TargetGene':TargetGene,'Tissue':tis,'Signor':str(PMID)}
                combined_df = pd.concat([combined_df,pd.DataFrame([newrow])], ignore_index=True)


In [1222]:
combined_df

,TF,TargetGene,Tissue,Experiment,Chea,Signor,Trrust
0,FOXA1,AKR1C1,prostate,"FOXA1 ChIP-Seq, CON KD | FOXA1 ChIP-Seq, CREB1...",,,
1,YY1,AKR1C1,lung,YY1 IP - A549 cells,,,
2,FOXA1,AKR1C1,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, FOXA1 ChIP-Seq -...",,,
3,DEX,AKR1C1,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, DEX | FOXA1 ChIP...",,,
4,NFYB,AKR1C1,leukocytes,NFYB IP - K562 cells,,,
...,...,...,...,...,...,...,...
2249,NR5A1,CYP19A1,uterus,NaN,NaN,19022561,NaN
2250,FOS,CYP19A1,uterus,NaN,NaN,19022561,NaN
2251,CREB1,CYP19A1,mammarygland,NaN,NaN,15955695,NaN
2252,ESRRA,CYP19A1,mammarygland,NaN,NaN,15955695,NaN


Trrust

In [1223]:
Trrust_path = f'D:/Raylab/LiMeNEx/FetchData/data/DikshaFiles/{folderName}/trrust/{folderName} - Trrust.csv'
TrrDf = pd.read_csv(Trrust_path)

In [1224]:
TrrDf.columns

Index(['TargetGene', 'TF', 'Library/PMID', 'Tissue', 'Paper Links',
       'Paper_link'],
      dtype='object')

In [1225]:
TrrDf.head()

,TargetGene,TF,Library/PMID,Tissue,Paper Links,Paper_link
0,CYP11A1,NR4A1,20083153,Ovary,https://pubmed.ncbi.nlm.nih.gov/20083153/,https://www.sciencedirect.com/science/article/...
1,CYP11A1,NR5A1,23530236,Ovary,https://pubmed.ncbi.nlm.nih.gov/20083153/,https://www.sciencedirect.com/science/article/...
2,CYP11A1,NR5A1,11057754,Liver,https://pubmed.ncbi.nlm.nih.gov/11057754/,https://www.sciencedirect.com/science/article/...
3,CYP11A1,NR5A2,19067654,Liver,https://pubmed.ncbi.nlm.nih.gov/19067654/,https://portlandpress.com/biochemj/article-abs...
4,CYP11A1,SF1,19477906,Liver,https://pubmed.ncbi.nlm.nih.gov/19477906/,https://jme.bioscientifica.com/view/journals/j...


In [1103]:
# list(TrrDf['validated genes'])

In [1226]:
# Trrust_path = f'D:/Raylab/LiMeNEx/FetchData/data/{folderName}/trrust/{folderName} - Trrust.csv'
# TrrDf = pd.read_csv(Trrust_path)
# TrrDf.head()
# TrrDf['validated genes'] = [var if pd.isna(var) else var.strip().lower() for var in list(TrrDf['validated genes'])]
# TrrDf = TrrDf[(TrrDf['validated genes'] == 'yes')]

for i in range(0,len(TrrDf)):
    tissueList = set([var.lower().replace(" ", "").strip() for var in TrrDf.iloc[i]['Tissue'].split(',')])

    redundantTissue = set(tissueList) - ActualTissue
    if len(redundantTissue) != 0:
        print("found redundant Tissue :", redundantTissue)
        break

    # TF = TrrDf.iloc[i]['TF']
    TF = set([var.replace(" ", "").strip() for var in TrrDf.iloc[i]['TF'].split(',')])
    TargetGene = TrrDf.iloc[i]['TargetGene']
    PMID = str(TrrDf.iloc[i]['Library/PMID'])

    if TargetGene not in ActualGenes:
        print("Redundant Gene :",TargetGene)
        continue

    for tis in tissueList:
        for tf in TF:

            mask = (combined_df['TF'] == tf) & (combined_df['TargetGene'] == TargetGene) & (combined_df['Tissue'] == tis)
            temp = combined_df[mask]

            if(len(temp) != 0):
                if(len(temp) > 1):
                    print('Here')
                index = temp.index[0]
                if pd.isna(combined_df.at[index, 'Trrust']) or combined_df.at[index, 'Trrust'] == '':
                    combined_df.at[index, 'Trrust'] = f"{str(PMID)}"
                else:
                    combined_df.at[index, 'Trrust'] += f";{str(PMID)}"
                    # print(combined_df.at[index,'TargetGene'],combined_df.at[index, 'Trrust'])
                    
            else:
                newrow = {"TF":tf, 'TargetGene':TargetGene,'Tissue':tis,'Trrust':str(PMID)}
                combined_df = pd.concat([combined_df,pd.DataFrame([newrow])], ignore_index=True)


In [1227]:
combined_df

,TF,TargetGene,Tissue,Experiment,Chea,Signor,Trrust
0,FOXA1,AKR1C1,prostate,"FOXA1 ChIP-Seq, CON KD | FOXA1 ChIP-Seq, CREB1...",,,
1,YY1,AKR1C1,lung,YY1 IP - A549 cells,,,
2,FOXA1,AKR1C1,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, FOXA1 ChIP-Seq -...",,,
3,DEX,AKR1C1,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, DEX | FOXA1 ChIP...",,,
4,NFYB,AKR1C1,leukocytes,NFYB IP - K562 cells,,,
...,...,...,...,...,...,...,...
2324,NR5A1,CYP19A1,ovary,NaN,NaN,NaN,21087664
2325,NR5A1,CYP19A1,liver,NaN,NaN,NaN,21087664
2326,NR5A1,CYP19A1,mammarygland,NaN,NaN,NaN,21087664
2327,NR5A2,CYP19A1,liver,NaN,NaN,NaN,19067654


Chea

In [1228]:
cheaPath = F'D:/Raylab/LiMeNEx/FetchData/data/DikshaFiles/{folderName}/chea/{folderName} - Chea.csv'
cheaDf = pd.read_csv(cheaPath)
cheaDf.head()


,TF,Overlapping_Genes,Pubmed_ID,Pubmed_Link,Genes mentioned in the paper,Tissue,Paper Link
0,VDR,"CYP19A1,HSD17B2",23401126,https://pubmed.ncbi.nlm.nih.gov/23401126/,"CYP19A1,HSD17B2","Leukocytes, Blood",https://journals.sagepub.com/doi/10.1177/13524...
1,RELA,"AKR1C1,CYP7B1,CYP19A1,HSD11B2",24523406,https://pubmed.ncbi.nlm.nih.gov/24523406/,"AKR1C1,CYP7B1,CYP19A1,HSD11B2",Lung,https://www.sciencedirect.com/science/article/...
2,TP53,"AKR1D1,HSD17B3",16413492,https://pubmed.ncbi.nlm.nih.gov/16413492/,"AKR1D1,HSD17B3",Colon,https://www.sciencedirect.com/science/article/...
3,AR,"SRD5A1,HSD17B3,AKR1C2,CYP7B1,CYP19A1",19668381,https://pubmed.ncbi.nlm.nih.gov/19668381/,"SRD5A1,HSD17B3,AKR1C2,CYP7B1,CYP19A1",Prostate,https://journals.plos.org/plosone/article?id=1...
4,FOXA2,"AKR1D1,AKR1C2,CYP19A1,HSD17B2",19822575,https://pubmed.ncbi.nlm.nih.gov/19822575/,"AKR1D1,AKR1C2,CYP19A1,HSD17B2",Liver,https://academic.oup.com/nar/article/37/22/749...


In [1229]:
cheaDf.columns

Index(['TF', 'Overlapping_Genes', 'Pubmed_ID', 'Pubmed_Link',
       'Genes mentioned in the paper', 'Tissue', 'Paper Link'],
      dtype='object')

In [1230]:
cheaDf['Genes mentioned in the paper'] = [var.strip() for var in list(cheaDf['Genes mentioned in the paper'])]
cheaDf = cheaDf[(cheaDf['Genes mentioned in the paper'] != 'not found')]

In [1231]:
for i in range(0,len(cheaDf)):

    PMID = str(cheaDf.iloc[i]['Pubmed_ID'])

    # print(PMID)
    # print(cheaDf.iloc[i]['Tissue'])
    tissueList = set([tissue.lower().replace(" ", "").strip() for tissue in cheaDf.iloc[i]['Tissue'].split(',')])
    TF = cheaDf.iloc[i]['TF']
    TargetGene = set([gene.replace(" ", "").strip() for gene in cheaDf.iloc[i]['Genes mentioned in the paper'].split(',')])

    redundantGene = set(TargetGene) - ActualGenes
    if len(redundantGene) != 0:
        print("Redundant Gene :",redundantGene)
        continue

    redundantTissue = set(tissueList) - ActualTissue
    if len(redundantTissue) != 0:
        print("found redundant Tissue :", redundantTissue)
        break

    for tis in tissueList:
        for TG in TargetGene:
            
            mask = (combined_df['TF'] == TF) & (combined_df['TargetGene'] == TG) & (combined_df['Tissue'] == tis)
            temp = combined_df[mask]

            if(len(temp) != 0):
                if(len(temp) > 1):
                    print('Here')
                index = temp.index[0]
                if pd.isna(combined_df.at[index, 'Chea']) or combined_df.at[index, 'Chea'] == '':
                    combined_df.at[index, 'Chea'] = f"{str(PMID)}"
                else:
                    combined_df.at[index, 'Chea'] += f";{str(PMID)}"
            else:
                newrow = {"TF":TF, 'TargetGene':TG,'Tissue':tis,'Chea':str(PMID)}
                combined_df = pd.concat([combined_df,pd.DataFrame([newrow])], ignore_index=True)
            
    

In [1232]:
combined_df.to_csv(f'D:/Raylab/LiMeNEx/sbmlData/pathwayTfsModified/{folderName}.csv',index=False)

In [1233]:
combined_df['Tissue'].unique()

array(['prostate', 'lung', 'mammarygland', 'leukocytes', 'stomach',
       'adiposetissue', 'stemcells', 'liver', 'uterus', 'skin',
       'vasculature', 'pancreas', 'hspcs', 'embryonictissue', 'colon',
       'cns', 'skeletalmuscle', 'megakaryocytelineage', 'esophagus',
       'kidney', 'heart', 'miscellaneous', 'bone', 'sensory', 'placenta',
       'ovary', 'adrenalgland', 'testis', 'fibroblasts', 'blood',
       'erythroidlineage'], dtype=object)

In [1112]:
combined_df

,TF,TargetGene,Tissue,Experiment,Chea,Signor,Trrust
0,FOXA1,AGMO,prostate,"FOXA1 ChIP-Seq, CON KD | FOXA1 ChIP-Seq, CON K...",,,
1,CTCF,AGMO,skin,"CTCF IP - HFFC cells, Rep 2 , CTCF IP - BJ cel...",,,
2,CDX2,AGMO,colon,"CDX2 IP | 125DVD3, CDX2 IP",,,
3,125DVD3,AGMO,colon,"CDX2 IP | 125DVD3, CEBPB IP | 125DVD3",,,
4,FOXA1,AGMO,mammarygland,"DEX | FOXA1 ChIP-Seq - Rep 2, FOXA1 ChIP-Seq -...",,,
...,...,...,...,...,...,...,...
4075,ELF5,LPIN2,mammarygland,NaN,23300383,NaN,NaN
4076,TP63,AGPAT2,prostate,NaN,23658742,NaN,NaN
4077,SMAD3,AGPAT3,embryonictissue,NaN,21741376,NaN,NaN
4078,TFAP2A,DGAT2,mammarygland,NaN,17053090,NaN,NaN


Assigning tissue to tfs ineraction

In [3]:
import pandas as pd
tfs = pd.read_csv('D:/Raylab/LiMeNEx/FetchData/final_tfs_interaction_mod.csv')

In [4]:
tfs.head()

,TF,TargetGene,Effect,PMID,DATABASEA
0,AR,AKR1C3,Repression,22971343,UNIPROT
1,CREBBP,ALOX15,Activation,12517954,UNIPROT
2,EP300,ALOX15,Activation,12517954,UNIPROT
3,SP1,ALOX5,Activation,19781662,UNIPROT
4,MECP2,ALOX5,Repression,19781662,UNIPROT


In [5]:
tfs['Tissue'] = ""
tfs.head()

,TF,TargetGene,Effect,PMID,DATABASEA,Tissue
0,AR,AKR1C3,Repression,22971343,UNIPROT,
1,CREBBP,ALOX15,Activation,12517954,UNIPROT,
2,EP300,ALOX15,Activation,12517954,UNIPROT,
3,SP1,ALOX5,Activation,19781662,UNIPROT,
4,MECP2,ALOX5,Repression,19781662,UNIPROT,


In [15]:
import os
import pandas as pd
from collections import defaultdict
import json

pathwaydict = {}
folder_path = 'D:/Raylab/LiMeNEx/sbmlData/pathwayTfsModified'

unique_pairs = set()
tissue_counts = defaultdict(int)  # To store occurrences of each tissue
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        # Construct the full file path
        file_path = os.path.join(folder_path, filename)
        
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path)

        pathwaydict[filename] = set(list(df['TargetGene'].unique()))
        unique_pairs.update(zip(df['TF'], df['TargetGene']))

        for tissue in df['Tissue']:
            tissue_counts[tissue] += 1
        

tissue_counts_dict = dict(tissue_counts)

# Save tissue counts to a JSON file
json_path = "D:/Raylab/LiMeNEx/FetchData/tissue_counts.json"
with open(json_path, "w") as json_file:
    json.dump(tissue_counts_dict, json_file, indent=4)

total_unique_pairs = len(unique_pairs)
print("Total unique (tfs, enzymaticGene) pairs:", total_unique_pairs)


Total unique (tfs, enzymaticGene) pairs: 31551


In [1239]:
len(pathwaydict)

22

In [1182]:
tfs['DATABASEA'].unique()

array(['UNIPROT', 'SIGNOR', 'Trrust'], dtype=object)

In [1247]:
tfs

,TF,TargetGene,Effect,PMID,DATABASEA,Tissue
0,AR,AKR1C3,Repression,22971343,UNIPROT,prostate
1,CREBBP,ALOX15,Activation,12517954,UNIPROT,lung
2,EP300,ALOX15,Activation,12517954,UNIPROT,lung
3,SP1,ALOX5,Activation,19781662,UNIPROT,"leukocytes,blood,hspcs"
4,MECP2,ALOX5,Repression,19781662,UNIPROT,leukocytes
...,...,...,...,...,...,...
525,NFYC,LPIN1,Unknown,19553673,Trrust,"adiposetissue,skeletalmuscle,liver"
526,SREBF1,LPIN1,Unknown,19553673,Trrust,"liver,skeletalmuscle,adiposetissue"
527,SREBF2,ACOT7,Activation,16335799,Trrust,cns
528,RARA,SCD,Activation,11397803,Trrust,sensory


In [8]:
for i in range(0,len(tfs)):

    pathway = ""
    TargetGene = tfs.at[i,'TargetGene']
    TF = tfs.at[i,'TF']
    PMID = tfs.at[i,'PMID']
    PMID = set([temp.strip() for temp in PMID.split(';')])
    # database = tfs.at[i,'DATABASEA']
    tissue = []

    for key,val in pathwaydict.items():
        if TargetGene in val:
            pathway = key
    
            file_path = os.path.join('D:/Raylab/LiMeNEx/sbmlData/pathwayTfsModified', pathway)
            df = pd.read_csv(file_path,dtype={'Signor': str,'Chea':str,'Trrust':str})

            # df['Signor'] = df['Signor'].astype(str)
            # df['Trrust'] = df['Trrust'].astype(str)

            # dbColumnName = 'Signor' if database in ['UNIPROT','SIGNOR'] else 'Trrust'
            temp = df[(df['TargetGene'] == TargetGene) & (df['TF'] == TF)]

            for j in list(temp.index):
                val1 = temp.at[j,'Signor']
                val2 = temp.at[j,'Trrust']
                val = []

                if (pd.isna(val1) or val1 == "") and (pd.isna(val2) or val2 == ""):
                    continue
                elif (pd.isna(val1) or val1 == ""):
                    val = val2.split(';')
                elif (pd.isna(val2) or val2 == ""):
                    val = val1.split(';')
                else:
                    val = val1.split(';') + val2.split(';')
                
                val = set([id.strip() for id in val])
                inter = val & PMID

                if len(inter) > 0:
                    tissue.append(temp.at[j,'Tissue'])

    tissue = list(set(tissue))
    tfs.at[i,'Tissue'] = ",".join(tissue)



In [9]:
temp = tfs[~(tfs['Tissue'] == '')]

In [10]:
len(temp['PMID'].unique())

305

In [12]:
temp.to_csv('D:/Raylab/LiMeNEx/FetchData/foundTfs.csv',index = False)